# 06. Research Agent - Microsoft Agent Framework Edition

## 개요
웹 검색 기능을 통합한 연구 에이전트를 구축합니다.

### 학습 목표
- 웹 검색 도구 통합
- 실시간 정보 수집
- 정보 분석 및 요약
- 멀티 소스 데이터 통합

## 1. 환경 설정

In [1]:
import os
import asyncio
import requests
from dotenv import load_dotenv
from typing import Annotated, List, Dict, Any
from pydantic import Field

from agent_framework import ChatAgent
from agent_framework.azure import AzureOpenAIChatClient

# 환경변수 로드
load_dotenv()

# Azure OpenAI Chat Client 생성
chat_client = AzureOpenAIChatClient(
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
)

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


## 2. 웹 검색 도구 구현

SerpAPI를 사용한 웹 검색 기능을 구현합니다.

In [2]:
def search_web(
    query: Annotated[str, Field(description="Search query")],
    num_results: Annotated[int, Field(description="Number of results", ge=1, le=10)] = 5
) -> str:
    """Search the web using SerpAPI and return results."""
    
    serp_api_key = os.getenv("SERP_API_KEY")
    
    if not serp_api_key:
        # SerpAPI가 없으면 모의 결과 반환
        return f"""Search results for '{query}':
        
1. Microsoft Agent Framework Overview
   - Microsoft Agent Framework is a next-generation SDK for building AI agents
   - Combines strengths of AutoGen and Semantic Kernel
   - Source: Microsoft Learn

2. Getting Started with Agent Framework
   - Install with: pip install agent-framework --pre
   - Supports both Python and .NET
   - Source: GitHub Documentation

3. Multi-Agent Workflows
   - Graph-based orchestration for complex tasks
   - Built-in checkpointing and state management
   - Source: Agent Framework Blog
        """
    
    try:
        # SerpAPI 호출
        params = {
            "q": query,
            "api_key": serp_api_key,
            "num": num_results,
            "engine": "google"
        }
        
        response = requests.get("https://serpapi.com/search", params=params, timeout=10)
        data = response.json()
        
        # 결과 포맷팅
        results = []
        for i, result in enumerate(data.get("organic_results", [])[:num_results], 1):
            results.append(f"""{i}. {result.get('title', 'N/A')}
   {result.get('snippet', 'No description')}
   Source: {result.get('link', 'N/A')}""")
        
        return f"Search results for '{query}':\n\n" + "\n\n".join(results)
        
    except Exception as e:
        return f"Search failed: {str(e)}"

print("✅ 웹 검색 도구 정의 완료")

✅ 웹 검색 도구 정의 완료


## 3. 연구 에이전트 생성

In [3]:
# 웹 검색 기능을 가진 연구 에이전트
research_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Research Specialist with web search capabilities.
    
    Your role:
    1. Use web search to find current, accurate information
    2. Analyze and synthesize information from multiple sources
    3. Provide well-structured, cited responses
    4. Identify key insights and trends
    
    Always cite your sources and provide links when available.
    """,
    tools=[search_web],
    name="ResearchAgent"
)

print("✅ 연구 에이전트 생성 완료")

✅ 연구 에이전트 생성 완료


## 4. 기본 연구 작업

In [4]:
async def basic_research():
    """기본 웹 검색 연구"""
    
    query = "What are the latest developments in Microsoft Agent Framework?"
    
    print(f"\n{'='*60}")
    print(f"Research Query: {query}")
    print(f"{'='*60}\n")
    
    result = await research_agent.run(query)
    
    print("📊 Research Results:\n")
    print(result.text)
    print(f"\n{'='*60}\n")

await basic_research()


Research Query: What are the latest developments in Microsoft Agent Framework?

📊 Research Results:

The **Microsoft Agent Framework** has seen several important developments recently, making it a prominent tool in building AI agents and enhancing multi-agent workflows. Here are the key highlights:

### 1. **General Availability and Open Source Launch**
The Microsoft Agent Framework was released for general availability on **October 2, 2025**. It is an open-source development kit designed for building AI agents, which can operate in .NET and Python environments. This framework is built to streamline the process of coding AI agents by reducing complexity and facilitating multi-agent workflows (source: [Microsoft Docs](https://learn.microsoft.com/en-us/agent-framework/overview/agent-framework-overview)).

### 2. **Integration with Microsoft Foundry**
In late 2025, Microsoft also updated its Foundry platform to allow the deployment of custom code agents created with the Microsoft Agent F

## 5. 비교 연구

In [5]:
async def comparative_research():
    """여러 주제를 비교 연구"""
    
    query = """Compare the following AI agent frameworks:
    1. Microsoft Agent Framework
    2. LangChain
    3. AutoGen
    
    Focus on:
    - Key features
    - Use cases
    - Strengths and weaknesses
    """
    
    print(f"\n{'='*60}")
    print("Comparative Research")
    print(f"{'='*60}\n")
    
    result = await research_agent.run(query)
    
    print("📊 Comparison Results:\n")
    print(result.text)
    print(f"\n{'='*60}\n")

await comparative_research()


Comparative Research

📊 Comparison Results:

Here’s a comparative analysis of three AI agent frameworks: Microsoft Agent Framework, LangChain, and AutoGen, focusing on their key features, use cases, strengths, and weaknesses.

### 1. Microsoft Agent Framework

**Key Features:**
- Open-source development kit for building AI agents.
- Supports .NET and Python for multi-agent workflows.
- Integration with Azure services for cloud-native applications.

**Use Cases:**
- Optimizing business processes through automation.
- Creating chatbots and virtual assistants.
- Developing applications that require complex decision-making and workflows.

**Strengths:**
- Rich integration with Microsoft’s Azure infrastructure allows for scalability and reliability.
- Simplifies coding processes, making it approachable for developers with diverse skill sets.
- Boosts operational efficiency by handling complex tasks with fewer resources.

**Weaknesses:**
- Primarily targeted at users within the Microsoft ec

## 6. 멀티 에이전트 연구 시스템

여러 전문가 에이전트가 협력하여 종합적인 연구를 수행합니다.

In [6]:
# 추가 전문가 에이전트들
technical_analyst = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Technical Analyst.
    Analyze technical aspects, architecture, and implementation details.
    Focus on technical feasibility and requirements.
    """,
    name="TechnicalAnalyst"
)

market_analyst = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Market Analyst.
    Analyze market trends, adoption rates, and competitive landscape.
    Focus on business value and market opportunities.
    """,
    name="MarketAnalyst"
)

report_writer = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a Report Writer.
    Synthesize information from multiple sources into comprehensive reports.
    Create clear, well-structured documents with executive summaries.
    """,
    name="ReportWriter"
)

print("✅ 전문가 에이전트 팀 생성 완료")

✅ 전문가 에이전트 팀 생성 완료


In [7]:
async def collaborative_research(topic: str):
    """협업 연구: Research → Technical Analysis → Market Analysis → Report"""
    
    print(f"\n{'='*60}")
    print(f"Collaborative Research: {topic}")
    print(f"{'='*60}\n")
    
    # Step 1: 웹 검색 기반 연구
    print("📚 Step 1: Web Research...")
    research_result = await research_agent.run(
        f"Research comprehensive information about: {topic}"
    )
    print(f"Research completed: {len(research_result.text)} characters\n")
    
    # Step 2: 기술 분석
    print("🔧 Step 2: Technical Analysis...")
    tech_result = await technical_analyst.run(
        f"""Provide technical analysis based on this research:
        
        {research_result.text}
        
        Focus on:
        - Architecture and design
        - Implementation considerations
        - Technical requirements
        """
    )
    print(f"Technical analysis completed\n")
    
    # Step 3: 시장 분석
    print("📊 Step 3: Market Analysis...")
    market_result = await market_analyst.run(
        f"""Provide market analysis based on this research:
        
        {research_result.text}
        
        Focus on:
        - Market trends
        - Adoption potential
        - Competitive positioning
        """
    )
    print(f"Market analysis completed\n")
    
    # Step 4: 종합 보고서 작성
    print("📝 Step 4: Report Generation...")
    final_report = await report_writer.run(
        f"""Create a comprehensive report integrating these analyses:
        
        RESEARCH FINDINGS:
        {research_result.text}
        
        TECHNICAL ANALYSIS:
        {tech_result.text}
        
        MARKET ANALYSIS:
        {market_result.text}
        
        Include:
        1. Executive Summary
        2. Key Findings
        3. Technical Overview
        4. Market Insights
        5. Recommendations
        """
    )
    
    print(f"\n{'='*60}")
    print("FINAL RESEARCH REPORT")
    print(f"{'='*60}\n")
    print(final_report.text)
    print(f"\n{'='*60}\n")
    
    return final_report.text

# 실행
report = await collaborative_research(
    "AI agents in enterprise customer service automation"
)


Collaborative Research: AI agents in enterprise customer service automation

📚 Step 1: Web Research...
Research completed: 4611 characters

🔧 Step 2: Technical Analysis...
Technical analysis completed

📊 Step 3: Market Analysis...
Market analysis completed

📝 Step 4: Report Generation...

FINAL RESEARCH REPORT

# Comprehensive Report: AI Agents in Enterprise Customer Service Automation

## Executive Summary
AI agents are revolutionizing enterprise customer service by automating interactions, enhancing personalization, and improving operational efficiency. The integration of technologies like machine learning and natural language processing enables 24/7 service availability and human-like interactions, meeting growing customer demands. While the benefits, including cost reduction and improved customer insights, are clear, companies must navigate implementation challenges, data privacy concerns, and the need for human oversight. This report synthesizes research findings, technical analy

## 7. 추가 도구: 날씨, 뉴스, 환율

In [8]:
def get_weather(
    location: Annotated[str, Field(description="City name or location")]
) -> str:
    """Get current weather for a location."""
    
    api_key = os.getenv("OPENWEATHER_API_KEY")
    
    if not api_key:
        # Mock data
        return f"Weather in {location}: Partly cloudy, 22°C, Humidity: 65%"
    
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": location,
            "appid": api_key,
            "units": "metric"
        }
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        temp = data["main"]["temp"]
        description = data["weather"][0]["description"]
        humidity = data["main"]["humidity"]
        
        return f"Weather in {location}: {description}, {temp}°C, Humidity: {humidity}%"
    except Exception as e:
        return f"Weather data unavailable: {str(e)}"

def get_exchange_rate(
    from_currency: Annotated[str, Field(description="Source currency (e.g., USD)")],
    to_currency: Annotated[str, Field(description="Target currency (e.g., KRW)")]
) -> str:
    """Get current exchange rate between two currencies."""
    
    # Mock data for demonstration
    rates = {
        ("USD", "KRW"): 1320.50,
        ("USD", "EUR"): 0.92,
        ("USD", "JPY"): 149.80,
        ("EUR", "USD"): 1.09,
        ("KRW", "USD"): 0.00076,
    }
    
    rate = rates.get((from_currency, to_currency), 1.0)
    return f"1 {from_currency} = {rate:.4f} {to_currency}"

# 다양한 도구를 가진 통합 에이전트
multi_tool_agent = ChatAgent(
    chat_client=chat_client,
    instructions="""
    You are a versatile assistant with access to:
    - Web search
    - Weather information
    - Exchange rates
    
    Use the appropriate tools to answer user queries accurately.
    """,
    tools=[search_web, get_weather, get_exchange_rate],
    name="MultiToolAgent"
)

print("✅ 다중 도구 에이전트 생성 완료")

✅ 다중 도구 에이전트 생성 완료


In [9]:
async def multi_tool_demo():
    """여러 도구를 사용하는 복합 쿼리"""
    
    queries = [
        "What's the weather in Seoul and the current USD to KRW exchange rate?",
        "Search for recent AI news and tell me the weather in San Francisco",
        "What's the exchange rate from EUR to USD?"
    ]
    
    for query in queries:
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print(f"{'='*60}\n")
        
        result = await multi_tool_agent.run(query)
        print(f"Response:\n{result.text}\n")

await multi_tool_demo()


Query: What's the weather in Seoul and the current USD to KRW exchange rate?

Response:
I couldn't retrieve the current weather data for Seoul, but I have the exchange rate for USD to KRW. 

The current exchange rate is:
- 1 USD = 1,320.50 KRW. 

If you need more information or specifics about the weather, please let me know!


Query: Search for recent AI news and tell me the weather in San Francisco

Response:
### Recent AI News
1. **Nvidia-Backed AI Startup Synthesia Raises Funding**  
   Artificial-intelligence company Synthesia has raised $200 million, achieving a valuation of $4 billion. [Read more](https://www.wsj.com/tech/ai?gaa_at=eafs&gaa_n=AWEtsqd8eJFrN0WEAeSrOvqi95beFEHRVhR94diEiWNZEtwjo7HKUGo9yYwJ&gaa_ts=69774170&gaa_sig=DT1myIMlccdFBiyuNXlT5AEsNRUQ-gYZkk7NbgshvG9M1xmJCITny2TTeyarkOfX-ntNZDEIAPnvPVVbgUFGag%3D%3D).

2. **AI News - Latest Insights**  
   A compilation of the latest updates in artificial intelligence, machine learning, deep learning, and emerging technology w

## 마무리

### 학습한 내용
1. ✅ 웹 검색 도구 통합
2. ✅ 실시간 정보 수집
3. ✅ 멀티 에이전트 연구 시스템
4. ✅ 다양한 외부 API 통합
5. ✅ 협업 연구 워크플로우
6. ✅ 종합 보고서 생성

### 핵심 개념
- **도구 통합**: Pydantic Field를 사용한 타입 안전한 도구 정의
- **외부 API**: REST API 호출 및 에러 핸들링
- **정보 통합**: 여러 소스의 데이터를 하나로 통합
- **협업 패턴**: 전문가 에이전트들의 순차적 협업

### 다음 단계
- `05_Code_Interpreter.ipynb`: AI 코드 생성 및 실행